# Importing the SparkSession
## SparkSession:
- It is an entry point for spark application

In [4]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder\
                    .master("local")\
                    .appName("test")\
                    .getOrCreate()

In [6]:
spark

# Dataframe Reader
```python
df = spark.read.format("format").options(**options).load("path")

# format --> CSV, Json, parquet..
# option --> inferschema, mode, header (optional)
# load --> Path where our data is residing
```

## Mode
- When reading files, mode defines `how to handle bad or malformed` records.
- There are 3 types of mode
    - **Failfast** --> fail executionif malformed record in dataset.
    - **Dropmalformed** --> Drop the corrupted record.
    - **Permissive** --> Default --> set null value to all corrupted field.

--------------

## Reading CSV file in Spark


In [7]:
flight_df = spark.read.format("csv")\
                .option("header", "false")\
                .option("inferschema", "false")\
                .option("mode","FAILFAST")\
                .load("data/2010_summary_flight_data.csv")

In [8]:
flight_df.show(5)

+-----------------+-------------------+-----+
|              _c0|                _c1|  _c2|
+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
+-----------------+-------------------+-----+
only showing top 5 rows



In [9]:
flight_df_header = spark.read.format("csv")\
                .option("header", "true")\
                .option("inferschema", "false")\
                .option("mode","FAILFAST")\
                .load("data/2010_summary_flight_data.csv")

flight_df_header.show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows



In [10]:
# printing inferschema with inferschema false
flight_df_header.printSchema() # all cols are reading as string value

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: string (nullable = true)



In [11]:
flight_df_header_schema = spark.read.format("csv")\
                .option("header", "true")\
                .option("inferschema", "true")\
                .option("mode","FAILFAST")\
                .load("data/2010_summary_flight_data.csv")

flight_df_header_schema.show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows



In [12]:
# Pring the inferschema
flight_df_header_schema.printSchema()

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: integer (nullable = true)



# Possible interview que
1. How to create schema in pyspark ?
2. What are other ways to creating it ?
3. What is structfield and structype in schema ?
4. What if i have header my data

### We have 2 method for creating schema
1. structType and structField
2. DDL


**StructType** --> Which defines our structure of DF (list of struct field)   
**StructField** --> ("name", Type, nullable(True/False))


**DDL_schema** = "id integer, name string, count integer"


In [13]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [14]:
my_schema = StructType(
    [
        StructField("DEST_COUNTRY_NAME", StringType(), True),
        StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
        StructField("count", IntegerType(), True)
    ]
)

flight_df_header_Manualschema = spark.read.format("csv")\
                .option("header", "false")\
                .option("skipRows","1")\
                .option("inferschema", "false")\
                .schema(my_schema)\
                .option("mode","PERMISSIVE")\
                .load("data/2010_summary_flight_data.csv")

flight_df_header_Manualschema.show(5)
# .option("skipRows","1") --> currently not running

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME| NULL|
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
+-----------------+-------------------+-----+
only showing top 5 rows



**Click on the below link for spark csv file read documentation page**
- [spark.apache.org/docs/latest/sql-data-sources-csv.html](https://spark.apache.org/docs/latest/sql-data-sources-csv.html)

# Handling corrupted records in spark
1. Have you worked with corrupted record ?
2. When do you say that its corrupted record ?
3. What happens when we encounter with corrupted records in different read mode ?
4. How can we print bad records ?
5. Where do you store corrupted records and how can we access it later ?

-------------

### USED CSV dataset
```csv
id,name,age,salary,address,nominee
1,Manish,26,75000,bihar,nominee1
2,Nikita,23,100000,uttarpradesh,nominee2
3,Pritam,22,150000,Bangalore,India,nominee3
4,Prantosh,17,200000,Kolkata,India,nominee4
5,Vikash,31,300000,,nominee5
```

-------------



In [15]:
# mode = PERMISSIVE --> 
# when it meets a corrupted record, puts the malformed string into a field configured by columnNameOfCorruptRecord, and sets malformed fields to null.

employee_df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .option("mode", "PERMISSIVE")\
    .load("data/employee_data.csv")

employee_df.show()

+---+------+---+------+------------+--------+
| id|  name|age|salary|     address| nominee|
+---+------+---+------+------------+--------+
|  1|Manish| 26| 75000|       bihar|nominee1|
|  2|Nikita| 23|100000|uttarpradesh|nominee2|
|  5|Vikash| 31|300000|        NULL|nominee5|
+---+------+---+------+------------+--------+



In [16]:
# mode = DROPMALFORMED -->
# ignores the whole corrupted records. This mode is unsupported in the CSV built-in functions

employee_df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .option("mode", "DROPMALFORMED")\
    .load("data/employee_data.csv")

employee_df.show()

+---+------+---+------+------------+--------+
| id|  name|age|salary|     address| nominee|
+---+------+---+------+------------+--------+
|  1|Manish| 26| 75000|       bihar|nominee1|
|  2|Nikita| 23|100000|uttarpradesh|nominee2|
|  5|Vikash| 31|300000|        NULL|nominee5|
+---+------+---+------+------------+--------+



In [17]:
# mode = FAILFAST -->
# throws an exception when it meets corrupted records. it will retrun the error as we have malformed/corrupted records in the csv file.


employee_df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .option("mode", "FAILFAST")\
    .load("data/employee_data.csv")

employee_df.show()

+---+------+---+------+------------+--------+
| id|  name|age|salary|     address| nominee|
+---+------+---+------+------------+--------+
|  1|Manish| 26| 75000|       bihar|nominee1|
|  2|Nikita| 23|100000|uttarpradesh|nominee2|
|  5|Vikash| 31|300000|        NULL|nominee5|
+---+------+---+------+------------+--------+

